In [11]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("Traffic_Volume_Counts_20251111.csv")

# Fix date format (NYC is usually month/day/year)
df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")

# Identify hourly columns
hourly_cols = [col for col in df.columns if ":" in col]


# Melt wide → long
df_long = df.melt(
    id_vars=["ID", "SegmentID", "Roadway Name", "From", "To", "Direction", "Date"],
    value_vars=hourly_cols,
    var_name="Hour",
    value_name="Traffic_Count"
)
# Fixing issues with strings in traffic count by eliminating them
df_long["Traffic_Count"] = (
    df_long["Traffic_Count"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# Robust hour parser
def parse_hour(h):
    try:
        time_part = h.split('-')[0].strip()
        if not ("AM" in time_part or "PM" in time_part):
            suffix = h.split('-')[1].strip()[-2:]
            time_part = f"{time_part} {suffix}"
        return pd.to_datetime(time_part, format="%I:%M %p").hour
    except:
        return np.nan

df_long["Hour_num"] = df_long["Hour"].apply(parse_hour)

# Drop invalids
df_long = df_long.dropna(subset=["Date", "Hour_num"])

# Compute Datetime
df_long["Datetime"] = df_long["Date"] + pd.to_timedelta(df_long["Hour_num"], unit="h")

df_long.sort_values(by=["SegmentID", "Datetime"], inplace=True)

print("Rows after cleaning:", len(df_long))
print("Unique dates:", df_long["Date"].nunique())
print("Unique segments:", df_long["SegmentID"].nunique())
print(df_long.head())


Rows after cleaning: 1026144
Unique dates: 608
Unique segments: 1956
         ID  SegmentID Roadway Name           From        To Direction  \
12651   200        202  Main Street  Hoover Avenue  82 Drive        SB   
55407   200        202  Main Street  Hoover Avenue  82 Drive        SB   
98163   200        202  Main Street  Hoover Avenue  82 Drive        SB   
140919  200        202  Main Street  Hoover Avenue  82 Drive        SB   
183675  200        202  Main Street  Hoover Avenue  82 Drive        SB   

             Date           Hour  Traffic_Count  Hour_num            Datetime  
12651  2014-10-11  12:00-1:00 AM          305.0         0 2014-10-11 00:00:00  
55407  2014-10-11    1:00-2:00AM          222.0         1 2014-10-11 01:00:00  
98163  2014-10-11    2:00-3:00AM          137.0         2 2014-10-11 02:00:00  
140919 2014-10-11    3:00-4:00AM          138.0         3 2014-10-11 03:00:00  
183675 2014-10-11    4:00-5:00AM          114.0         4 2014-10-11 04:00:00  


In [ ]:
# Baseline model, linear regression

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Starting with preparing to split
X = df_long[["Hour_num"]]
y = df_long["Traffic_Count"]

# Still having weird issue with NAs? This is a precaution, may not be needed
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]
df_clean = df_long[mask]

# Splitting by SegmentID (street) to keep streets either in test or train, not both
segments = df_long["SegmentID"].unique()

train_segments, test_segments = train_test_split(segments, test_size=0.2, random_state=42)

print("Train segments:", len(train_segments))
print("Test segments:", len(test_segments))


# Extra precaution to avoid error
train_mask = df_clean["SegmentID"].isin(train_segments)
test_mask  = df_clean["SegmentID"].isin(test_segments)

X_train = X[train_mask]
y_train = y[train_mask]
X_test  = X[test_mask]
y_test  = y[test_mask]

# Now, data is split, so moving to the actual baseline model
lr = LinearRegression()
lr.fit(X_train,y_train)

lr_pred = lr.predict(X_test)
baseline_mse = mean_squared_error(y_test, lr_pred)


print("Baseline Model MSE:", baseline_mse)


Train segments: 1564
Test segments: 392
Baseline Model MSE: 440694.62067069154


In [ ]:
# start training an RNN